In [29]:
"""
Step 1  --->  Generate 20 chromosomes initially.
Step 2  --->  Calculate their fitness.
Step 3  --->  Select: ---> best chromosome and worst chromosome and Second Worst
chromosome according to their fitness.
Step 4  --->  Crossover worst two chromosomes. (Most Worst and Second Worst)
Step 5  --->  Mutate one Most Worst offspring.
Step 6  --->  Create next generation using: ---> best chromosome ,
one normal offspring (Other Than Mutated) , one mutated offspring
(from most worst chrmosome fitness).
Step 7  --->  Repeat for many generations ( loop From Step2 ) .
Step 8  --->  Print final best solution.

Because over generations ---> bad chromosomes disappear and good chromosomes
survive and new combinations appear

Eventually the algorithm finds a chromosome with maximum coverage.

"""

import random     # imports Python’s random module.

GRID_SIZE = 6     # 6 x 6 Grid
NUM_ANTENNAS = 5  # each chromosome (set of coordinates of unique_positions or
                  # order pairs) has 5 antennas means to store 5 order pairs or
                  # 5 coordinates of unique_positions


def generate_chromosome():
    """
    Create one chromosome with 5 antenna unique_positions.
    Each position is (x, y), where x and y are from 1 to 6.
    """

    # We use a set because a set automatically avoids duplicates.
    unique_positions = set()

    # Repeat until we get 5 unique positions.
    while len(unique_positions) < NUM_ANTENNAS:

        # Pick a random x and random y between 1 and 6.

        x = random.randint(1, GRID_SIZE)
        y = random.randint(1, GRID_SIZE)

        # Add the position as a tuple (x, y) into the set.
        unique_positions.add((x, y))

        # Convert the set into a list and return it.
    return list(unique_positions)

# chromo is the single set of order paris from population
def calculate_fitness(chromo):

    # empty set to store all covered cells to avoid any duplication.
    covered_cells = set()

    for (x, y) in chromo: # Loop through every antenna in the chromosome.
                          # (x,y) --> (row, column)
        # own cell
        covered_cells.add((x, y))

        # up cell, makes sure we do not go outside the grid. SO if antenna is
        # already at top row, no upper cell is added.
        if x - 1 >= 1:
            covered_cells.add((x-1, y))

        # down cell, boundary check prevents going below the grid.
        if x + 1 <= GRID_SIZE:
            covered_cells.add((x+1, y))

        # left cell
        if y - 1 >= 1:
            covered_cells.add((x, y-1))

        # right cell
        if y + 1 <= GRID_SIZE:
            covered_cells.add((x, y+1))

    return len(covered_cells)

# Here we use rank-based selection. Because chromosomes are selected based on
# their rank after sorting.
# Here we sorts fitness:  ---> indexed_fitness.sort(key=lambda x: x[1])
# Then selects:  worst two and best one ---> So this is basically:
# Rank-based deterministic selection .You are choosing based on fitness ranking.

# ==============================================================================

# population --> the population list (all chromosomes).
# fitness --> list of fitness values for each chromosome.
# This function chooses: best chromosome and worst two chromosomes
def selection(population, fitness_list):
    """
    population =
    [
     C1, C2, C3, ..., C20
    ]

    fitness =
    [
     15, 18, 17, ..., 21
    ]

    Each fitness value corresponds to a chromosome in the same position.

    # enumerate(fitness) adds index with each fitness value. It pairs index +
    fitness value.
    chromosome index 0 -> fitness 15
    chromosome index 1 -> fitness 18
    chromosome index 2 -> fitness 17

        # e.g fitness = [18, 20, 19] becomes [(0,18), (1,20), (2,19)]
        # and then make that order pairs a list

    Enumerates are numerical/integral constants which assigns a indecies to the
    items     and returns an enumerate object that produces pairs in the form of
    (index, element). Means we named the itegral constants that 0 has named as
    West , 1 has named as East and so on like assigning the index to each item.
    Also by default indexing starts from 0 in enumeration but we can Starting
    from any number. enumerate(fitness, start=5):
    """

    fitness_index = list(enumerate(fitness_list))

    # sort by fitness Smallest fitness first, largest last.
    # here we use x[1] because we have to sort second part of order pair/tupple
    # as each element is (index, fitness) of fitness_index

    fitness_index.sort(key=lambda x: x[1])

    # First element of the sorted list. Chromosome 0 is the worst.
    # e.g fitness_index[0] = (0,15) ---> worst_index1 = 0 so we need only first
    # part of order pair means the row part(index of most worst fitness)
    # so we write [0][0]

    worst_index1 = fitness_index[0][0]

    # Second worst chromosome.
    # e.g fitness_index[1] = (2,17) ---> worst_index2 = 2 so we need only first
    # part of order pair means the row part(index of Second worst fitness)
    # so we write [1][0]

    worst_index2 = fitness_index[1][0]

    # best chromosome is the last one after sorting in ascending order
    # -1 means the last element in the list.
    # e.g fitness_index[-1] = (1,18) ---> best_idx = 1
    best_idx = fitness_index[-1][0]

    # Retrieve the actual chromosome
    chromosome1 = population[worst_index1]

    # Retrieve the second worst chromosome
    chromosome2 = population[worst_index2]

    # Retrieve the best chromosome
    best_chromosome = population[best_idx]

    return chromosome1, chromosome2, best_chromosome, worst_index1, worst_index2, best_idx

# Crossover mixes information from two parents.
# Without crossover, The algorithm would only keep existing chromosomes.
# It would never combine good traits.
# With crossover, Good genes from two parents combine to create a better child.
# Example  --> Parent 1: ---> [(2,2), (1,4), (3,5), (4,6), (5,6)]
# Parent 2:  ---> [(3,1), (5,2), (4,4), (6,3), (2,5)]
# Crossover result: ---> [(2,2), (1,4), (3,5), (6,3), (2,5)]
# Now this child may cover more grid cells.
# So crossover helps discover better solutions.

# ============================================

# Create 2 children from the 2 worst chromosomes.Here we use Single-point crossover
# Single-point crossover involves choosing a random point and swapping the genetic
# material from two parents at that point to create two offspring.
# Single-point crossover works like -->  Choose one crossover point --> Split
# both parents at that point --> Swap the parts
# In typical single-point crossover, the crossover point is random.

def crossover(chromosome1, chromosome2):

    # Here offspring means the Child and chromosome1 and chromosome2
    # are parents to create the child by taking genes from each parent.

    # Here we use some fix rule of splitting the parents to get mix genes from
    # both parents

    # Creates the first child
    # chromosome1[:3] means take the first 3 genes from chromosome1
    # chromosome2[3:] means take the last 2 genes from chromosome2
    # e.g chromosome1 = [a, b, c, d, e] chromosome2 = [p, q, r, s, t] then
    # parent1 = [a, b, c, s, t]

    child1 = chromosome1[:3] + chromosome2[3:]

    # Creates the Second child
    # chromosome2[:3] means take the first 3 genes from chromosome2
    # chromosome1[3:] means take the last 2 genes from chromosome1
    # e.g chromosome1 = [a, b, c, d, e] chromosome2 = [p, q, r, s, t] then
    # parent2 = [p, q, r, d, e]

    child2 = chromosome2[:3] + chromosome1[3:]

    return child1, child2

# Crossover can only mix existing genes. But Mutation introduces new variations.
# Example mutation rule in given problem: y = y - 1
# This moves antennas upward.   ---> y=y-1
# This may produce new coverage areas that crossover alone cannot create.
# Mutation works only on worst chromosome Because we want to protect good
# solutions. If we mutated the best chromosome, we might destroy it.
# So the algorithm does: Best chromosome --> preserved
# Worst chromosome -> modified
# This keeps improving the population.

# ======================================================================

# It takes one offspring as input --> offspring will be changed
# according to the mutation rule (depends upon problem or we can choose our own).
# We mutate the child that comes from the worst chromosome
# In code this is done here:
# if fitness[worst_index1] <= fitness[worst_index2]:
# Means --->  if chromosome1 is worse -> mutate offspring1/child1
# otherwise -> mutate offspring2 /child2 So only one of the two child is mutated.

def mutation(chromo):
    mutated_chromosome_list = [] # Empty list to store the new mutated chromosome.

 # loop through each antenna position in the chromosome
 # Each iteration gets one coordinate pair like: (2,2) , (1,4) , (3,5)
 # and splits it into:  x and y
    for (x, y) in chromo:
        new_y = y - 1 # moves the antenna upward by decreasing its y by 1

        # This prevents the antenna from going outside the grid.
        # Because the grid starts at row 1, not 0
        if new_y < 1:
            new_y = 1

        # Add the mutated antenna position to the new chromosome.
        mutated_chromosome_list.append((x, new_y))

    # After all 5 antennas are processed, return the mutated chromosome.
    return mutated_chromosome_list

# prints every chromosome in the population along with its fitness
def print_population(population, fitness):
  for i in range(len(population)):
    # Print chromosome one by one and all stored coordinates/tupples/order pairs
    # (population) and its fitness.
    # Example output: Chromosome 1 : [(2,2), (1,4), (3,5), (4,6), (5,6)]
    # Fitness = 18  , i + 1 is used so printing starts from 1 instead of 0.
    """
    {i+1:2} means print chromosome number in width 2
    {fitness[i]:2} means print fitness in width 2
    So output becomes aligned:
    Chromosome  1 : [(2, 2), (1, 4), (3, 5), (4, 6), (5, 6)]   Fitness = 18
    Chromosome  2 : [(1, 1), (2, 3), (4, 5), (5, 2), (6, 6)]   Fitness = 20
    Chromosome 10 : [(2, 1), (3, 3), (4, 4), (5, 5), (6, 2)]   Fitness = 19
    """

    print(f"Chromosome {i+1:2} : {population[i]}   Fitness = {fitness[i]:2}")


def main():
  """
seed() does NOT generate numbers.It simply sets the starting state of the random
number generator.
The random generator is a machine that produces a sequence of numbers
seed() decides where in that sequence you start

Random numbers are NOT truly random Python doesn’t use true randomness
(unless you use special libraries like time libraries).It uses a pseudo-random generator (PRNG).

That means: It follows a fixed mathematical formula to generate numbers.
So internally it works like: next_number = f(previous_number)

So: It starts from a value --> called the seed. Then keeps generating numbers
deterministically.  Role of seed(1)
When you do:   random.seed(1),  You are setting: initial_state = 1
Then the generator does:
state1 = f(1)
state2 = f(state1)
state3 = f(state2) ...
So the sequence becomes:  2, 5, 1, 3, 1, ...
Stays constant Because: Same seed → same starting state , Same formula → same outputs

The generator doesn’t store infinite numbers. Instead: It has a formula
That formula can produce a very long sequence (billions of values)
Seed → first value → second → third → ...
No infinity stored — just computed step by step.

randint(1, 6) does NOT magically pick numbers. It works like this:
Python internally generates numbers like: 0.13436424411240122, 0.8474337369372327

Get a raw random number (e.g. between 0 and 1) , Scale it to your range (1 to 6)
Conceptually:
r = random()        # 0 ≤ r < 1
x = int(r * 6) + 1  # convert to 1–6
You get the SAME numbers every run Because:

Seed sets the starting state --> random.seed(1) This means:  Internal state = 1

Now the generator will follow a fixed formula starting from 1.
Generator produces raw random numbers
Python internally generates numbers like:
0.13436424411240122
0.8474337369372327 ...
These come from a deterministic algorithm (Mersenne Twister).

First call → randint(1, 6)
Internally:   r = 0.13436424411240122  ,  x = int(r * 6) + 1
Now calculate:  r * 6 = 0.1343642441 × 6 ≈ 0.806
int(0.806) = 0   ---> x = 0 + 1 = 1  ,  So: x = 1

Second call → next number in sequence --> Next raw random number:
r = 0.8474337369372327
Now: r * 6 ≈ 5.084   --> int(5.084) = 5  ,  y = 5 + 1 = 6  So: y = 6
Final Output   1   6

Sequence:
seed(1)  0.134364 --> 1 ,  0.847433 --> 6
If run it again , will ALWAYS get: 1  6  , Because: Same seed -> same raw numbers
Same raw numbers -> same calculations,  Same calculations → same result
If seed change -->  random.seed(2)  Now sequence changes:
0.956034 → 6  ,  0.947827 → 6
Output:  6   6

randint() does NOT choose randomly each time It just converts the next number
in a fixed sequence

If We DON’T use seed() , just write:
import random
x = random.randint(1, 6)
print(x)

Python automatically does something like:
random.seed(current_time)
So: It uses current time (or system entropy) as the seed.

Why time makes it “look random”, --> Time is always changing:
10:01:01.12345
10:01:01.67891
10:01:02.00012

So each run:  Different seed  --> Different starting point --> Different sequence
Example
import random
import time
random.seed(time.time())
print(random.randint(1, 6))

Every run → different output Because:  seed = 1711262345.123 , seed = 1711262346.891
Different seeds → different sequences

Even with time:  It is STILL pseudo-random, Because internally:
seed --> deterministic formula --> sequence
So:  Same time seed --> same sequence , Different time --> different sequence

Without seed (default behavior)  --> Looks random  --> Good for games, simulations
With fixed seed  --> Reproducible results --> Good for debugging, testing

But is time truly random? --> No — time is predictable. It just changes enough to look random

For REAL randomness If want true randomness (secure):

import secrets
secrets.randbelow(6) + 1

or

import os
os.urandom(8)

These use:

OS entropy (keyboard, mouse, hardware noise, etc.)

Python’s random:  random.seed(1) Uses a formula Completely predictable if you know the seed
That’s why it’s called pseudo-random

 What is “REAL randomness”? --> Real randomness means:
Even if someone knows everything, they still cannot predict the next number
Example:
Coin flip in real life , Noise from hardware , Electrical signals

What is “OS entropy ? Your computer is constantly doing unpredictable things:
Mouse movements , Keyboard timing , CPU temperature , Disk activity , Network delays
These are not perfectly predictable. The OS collects this into something called:
Entropy pool (randomness pool)

os.urandom() — what it does
import os
os.urandom(8)

This means: “Give me 8 bytes of random data from the OS entropy pool”
Example output: b'\xa3\x91\x7f\x02\xcc\x19\x88\xef'

This is: NOT generated by a formula so you can't reverse
NOT predictable ---> Comes from real-world noise

secrets module (simpler + safer) ---> Python made secrets for easy secure randomness

import secrets
secrets.randbelow(6) + 1

This does: Uses OS entropy internally --> Converts it into a number (like randint)
random.randint(1,6) seed → formula → number ( Predictable if seed is known )

secrets.randbelow(6)  --> OS noise → entropy → number ( Unpredictable )

Why do we need this?  For security --> Example: Passwords , OTP codes , Encryption keys
If you used:  random.seed(1)   An attacker could guess your “random” numbers
  """
  random.seed(1)

  population = [] # empty list to store the sets of chromosomes

  # initial population of 20 chromosomes
  for i in range(20):
    chromosome = generate_chromosome() # generatre the chromosome one by one
    population.append(chromosome) # store the chromosome to make population

  # This sets how many 'x' generations the algorithm will run.
  # So the selection/crossover/mutation process repeats x times.
  generations = 3   # can be 500 or 700

  for g in range(1, generations + 1):
      print("\n" + "=" * 50) # makes the output easier to read.
      print("GENERATION", g) # prints a title for each generation.
      print("=" * 50) # makes the output easier to read.

      # calculates fitness for every chromosome in the current population
      """
      In Generation 1 ---> Initial population size = 20 and fitness size = 0
      After calculate fitness: population size = 20 and fitness size = 20
      Creating the next generation we do:
      population = [best_chromo, other_offspring, mutated_offspring]
      So after the first generation, your population size becomes 3, not 20.
      So we have to clear the fitness list for each generation

      If we Don't clear the fitness list then After First Generation fitness
      size = 20 and After Second Generation fitness size = old 20 values +
      3 new values = 23 values So Now population size = 3 fitness size = 23
      e.g In Selection Function

      --->    fitness_index = list(enumerate(fitness_list))

      [(0,15),(1,18),(2,17),(3,15),(4,12),......,(22,14)]
      Then it sorts and picks worst index: worst_index1 = 17
      But population only has: population = [0,1,2]
      So when Python executes: chromosome1 = population[17] it crashes with
      IndexError: list index out of range because index 17 does not exist.
      So Reset fitness at the start of every generation.
      """
      # Reset fitness list for every generation.
      fitness = []  # empty list to store the fitness for each chromosome set

      for i in range(len(population)):
        chromo = population[i]  # take chromosome one by one
        fit_value = calculate_fitness(chromo)  # calculate its fitness
        fitness.append(fit_value)  # store fitness in the list

    # prints the current population and their fitness values to see all
    # chromosomes in that generation.

      print("\nCurrent population:")
      print_population(population, fitness)

      # selection
      # It returns ---> chromosome1 = worst chromosome ,
      # chromosome2 = second worst chromosome , best_chromo = best chromosome
      # worst_index1 = index of worst chromosome ,
      # worst_index2 = index of second worst chromosome
      # best_idx = index of best chromosome
      chromosome1, chromosome2, best_chromo, worst_index1, worst_index2, best_idx = selection(population, fitness)

      # Print the best chromosome and its fitness.
      # This is the chromosome that survives unchanged.
      print("\nBest chromosome (goes to next generation):")
      print(f"Chromosome {best_idx + 1:2} : {best_chromo}   Fitness = {fitness[best_idx]:2}")

      # Print the two chromosomes chosen for crossover. where chromosome1 is
      # the most worst (smallest fitness) and chromosome2 is the second worst
      # (Second Smallest Fitness)
      print("\nWorst two selected for crossover:")
      print(f"Chromosome {worst_index1 + 1:2} : {chromosome1}   Fitness = {fitness[worst_index1]:2}")
      print(f"Chromosome {worst_index2 + 1:2} : {chromosome2}   Fitness = {fitness[worst_index2]:2}")

      # crossover
      # crossover function returns child1 in offspring1 and child2 in offspring2
      offspring1, offspring2 = crossover(chromosome1, chromosome2)

      # Print the newly created offspring.
      print("\nAfter crossover:")
      print("Offspring 1:", offspring1)
      print("Offspring 2:", offspring2)

      # mutation on child coming from worst chromosome
      # This checks which of the two selected chromosomes is worse.
      # Lower fitness means worse.
      # If first chromosome is worse or equal, mutate offspring1.

      if fitness[worst_index1] <= fitness[worst_index2]:
          # If the first parent was worse: mutate offspring1
          # keep offspring2 unchanged
          # Store them in: mutated_offspring and other_offspring

          mutated_offspring = mutation(offspring1)
          other_offspring = offspring2
          print("\nMutation applied to Offspring 1")

          # Otherwise: mutate offspring2 and keep offspring1 unchanged
      else:
          mutated_offspring = mutation(offspring2)
          other_offspring = offspring1
          print("\nMutation applied to Offspring 2")

          # Print the mutated chromosome.
      print("Mutated Offspring:", mutated_offspring)

      # next generation ---> creates the new population for the next generation.
      # Even though we started with 20 chromosomes, this line makes the new
      # generation contain only:
      # 1 best chromosome , 1 normal offspring (Other Than Mutated Offspring)
      # and 1 mutated offspring (from the most worst chromosome fitness)
      # So after the first generation, your population size becomes 3, not 20.
      # It becomes a 3-chromosome GA after the first iteration.

      population = [best_chromo, other_offspring, mutated_offspring]

      # Reset next_fitness list for every generation to remove old ones.
      # empty list to store the Next Generation fitness for each chromosome set
      next_fitness = []

      # Calculate fitness for the new population.
      # Since population now has only 3 chromosomes, this list will contain 3 fitness values.

      for i in range(len(population)):
        var = population[i]  # take updated chromosome one by one
        next_fit_value = calculate_fitness(var)  # calculate its fitness
        next_fitness.append(next_fit_value)  # store fitness in the list

      # Print the new population and its fitness values.

      print("\nNew population for next generation:")
      print_population(population, next_fitness)

  # final result
  # Empty list to store final Fitness Values for final population
  final_fitness = []
  for i in range(len(population)):
    chromo = population[i] # each set from population one by one
    # Calculate the fitness for each chromo or set
    final_fit_value = calculate_fitness(chromo)
    final_fitness.append(final_fit_value) # store fitness in the list

  # Find the index of the chromosome with highest fitness.
  """
  max(final_fitness) finds highest fitness value
  .index(...) finds where that value appears
  """
  best_final_idx = final_fitness.index(max(final_fitness))

  # Print final title.
  print("\n" + "=" * 50)
  print("FINAL BEST SOLUTION")
  print("=" * 50)

  # Print the final best chromosome and its fitness.

  print("Best Chromosome:", population[best_final_idx])
  print("Best Fitness:", final_fitness[best_final_idx])


"""
NOTE: Here we use strategy , that after first iteration population will remain
with 3 sets (best chromosome , one normal offspring (Other Than Mutated) ,
one mutated offspring).
If we want always 20 sets in the population after second , third , so on
generations then we use different strategy

Goal  ---> Population size = 20

Every generation must also end with 20 chromosomes.

A Simple Strategy (Recommended)
Step 1 — Evaluate Fitness  ---> Calculate fitness of all 20 chromosomes.
Step 2 — Sort by Fitness  Sort them from worst --> best (in Ascending order).
Step 3 — Preserve Best Chromosomes which is also called Elitism
Keep the best 2 chromosomes unchanged.   ---> C20 and C19
These go directly to the next generation.
Because mutation or crossover could destroy a good solution.
So we protect the best individuals.
Now we already have:   ---> Next population = 2 chromosomes
Step 4 — Remaining Chromosomes , We still need:  --->  20 − 2 = 18 chromosomes
These will come from crossover and mutation.
Step 5 — Select Parents for Crossover
Take the remaining 18 chromosomes.  Pair them like this:
C1  + C2 , C3  + C4 , C5  + C6 , C7  + C8 , C9  + C10 , C11 + C12 , C13 + C14
C15 + C16 , C17 + C18     That gives:   --->  9 pairs
Step 6 — Crossover --> Each pair produces 2 children.
Example:  Parent1: [a,b,c,d,e] , Parent2: [p,q,r,s,t]
Child1: [a,b,c,s,t]  ,  Child2: [p,q,r,d,e]  So: 9 pairs × 2 children = 18 children
Now: 2 best survivors  +  18 children  =  20 chromosomes
Population size remains constant.
Step 7 — Mutation should not be applied to every child.
Typical mutation rate:   5% – 20%
For simplicity you can do:  Mutate one child from each pair ,  So:
9 children mutated  and  9 children unchanged
Or even simpler:  mutate with probability 0.1 or mutate only the child which
is from most worst chromosome.
Example:
if random.random() < 0.1:
    child = mutation(child)
Final Population
              OR
mutate worst child from each pair
After crossover and mutation:  2 best survivors  +  18 children  = 20 chromosomes
Next generation = 20 again.

This Strategy Works because it balances exploration and preservation:
Elitism   --->  Keeps the best solutions.
Crossover --->  Combines good genes.
Mutation  --->  Introduces new possibilities.
Constant Population  --->  Maintains algorithm stability.

Elitism in Genetic Algorithms means:
Always keep the best chromosome unchanged in the next generation.
In simple words --->If a chromosome has the highest fitness (best solution),
we copy it directly to the next generation without changing it.

This protects the best solution from being lost during: crossover and mutation
Example  --->  Suppose fitness values are:
C1 = 12 , C2 = 18 , C3 = 15 , C4 = 21  <-- best
With elitism, chromosome C4 is copied directly to the next generation.
Next Generation starts with:  C4 (best preserved)
Then the rest of the population is created using:  crossover and mutation

Without elitism:   mutation might damage the best chromosome the algorithm could
lose the best solution found so far
With elitism:   the best solution always survives

"""

# This starts the program.
# Without this line, Python would only define the functions and do nothing.
main()


GENERATION 1

Current population:
Chromosome  1 : [(4, 4), (6, 4), (1, 4), (2, 5), (1, 3)]   Fitness = 17
Chromosome  2 : [(4, 4), (2, 1), (5, 1), (6, 4), (4, 1)]   Fitness = 17
Chromosome  3 : [(1, 1), (3, 6), (1, 6), (2, 5), (1, 3)]   Fitness = 14
Chromosome  4 : [(2, 4), (6, 1), (4, 6), (5, 1), (5, 2)]   Fitness = 16
Chromosome  5 : [(4, 4), (6, 2), (4, 3), (3, 2), (5, 2)]   Fitness = 16
Chromosome  6 : [(1, 2), (3, 1), (1, 4), (5, 6), (6, 6)]   Fitness = 16
Chromosome  7 : [(5, 4), (2, 3), (5, 6), (6, 6), (6, 3)]   Fitness = 16
Chromosome  8 : [(4, 4), (1, 4), (4, 5), (2, 6), (3, 5)]   Fitness = 16
Chromosome  9 : [(6, 2), (1, 4), (6, 6), (6, 3), (3, 5)]   Fitness = 18
Chromosome 10 : [(1, 2), (3, 4), (6, 5), (6, 1), (5, 4)]   Fitness = 18
Chromosome 11 : [(6, 2), (5, 5), (5, 4), (3, 6), (4, 1)]   Fitness = 20
Chromosome 12 : [(2, 1), (4, 5), (2, 5), (3, 5), (5, 2)]   Fitness = 20
Chromosome 13 : [(5, 5), (3, 4), (6, 1), (4, 6), (3, 6)]   Fitness = 16
Chromosome 14 : [(2, 4), (5, 